# 18 — Bilan global (3 datasets × baselines + 4 stratégies, base & Optuna)

Synthèse quantitative de tout le run GPU : tableaux moy ± std, précisions par modèle, frontière de Pareto coût/précision, et classement global. Lit `results_gpu_run/`.

## 1) Chargement

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

for _p in ['../src', 'src', './src', '../../src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import importlib
import analysis_utils as A
importlib.reload(A)

ROOT = A.find_results_root()
OUT = os.path.join(ROOT, "analysis")
os.makedirs(OUT, exist_ok=True)
df = pd.DataFrame(A.load_rows(ROOT))
print("RESULTS_ROOT =", ROOT)
print(f"{len(df)} entrees | datasets: {sorted(df.dataset.unique())} | modeles: {df.model.nunique()}")

RESULTS_ROOT = C:\VSCode\PF22_MultiFidelity_DL\results_gpu_run
33 entrees | datasets: ['Animals-10', 'Imagewoof', 'Intel'] | modeles: 11


## 2) Tableaux récapitulatifs (moyenne ± écart-type)

In [2]:
def fmt(m, s):
    return "—" if m is None else f"{m:.1f} ± {s:.1f}"

for ds in A.DATASETS:
    sub = df[df.dataset == ds]
    if sub.empty:
        continue
    t = pd.DataFrame({
        "Modèle": sub.model.values,
        "Test HF": [fmt(h, s) for h, s in zip(sub.HF, sub.HF_std)],
        "Test BF": [fmt(h, s) for h, s in zip(sub.BF, sub.BF_std)],
        "Test Mixte": [fmt(h, s) for h, s in zip(sub.Mixte, sub.Mixte_std)],
        "Coût données": [("—" if pd.isna(x) else f"{x:,.0f}") for x in sub.data_cost],
        "Coût total": [("—" if pd.isna(x) else f"{x:,.0f}") for x in sub.total_cost],
    }).set_index("Modèle")
    display(Markdown(f"### {ds}"))
    display(t)
    t.to_csv(os.path.join(OUT, f"summary_{ds}.csv"))
print("CSV sauvés dans", OUT)

### Animals-10

,Test HF,Test BF,Test Mixte,Coût données,Coût total
Modèle,,,,,
BL1 (HF),54.3 ± 2.2,27.8 ± 0.8,41.1 ± 1.1,"23,520","470,400"
BL2 (BF),61.3 ± 6.3,71.0 ± 1.5,66.2 ± 2.7,"21,213","424,260"
BL3 (Mixte),74.0 ± 3.0,72.9 ± 1.1,73.4 ± 2.0,"44,733","894,660"
S1 Transfer,77.7 ± 0.6,68.3 ± 0.5,73.0 ± 0.3,"44,733","400,230"
S1 +Optuna,78.1 ± 0.1,67.8 ± 0.8,72.9 ± 0.4,"44,733","435,447"
S2 CoTrain,63.8 ± 5.2,61.0 ± 4.2,62.4 ± 4.7,"44,733","486,720"
S2 +Optuna,73.2 ± 2.1,69.4 ± 2.3,71.3 ± 2.2,"44,733","579,040"
S3 Curriculum,76.5 ± 0.2,73.1 ± 0.1,74.8 ± 0.1,"44,733","421,265"
S3 +Optuna,77.0 ± 0.6,73.9 ± 0.2,75.4 ± 0.4,"44,733","658,232"


### Imagewoof

,Test HF,Test BF,Test Mixte,Coût données,Coût total
Modèle,,,,,
BL1 (HF),25.0 ± 4.1,22.1 ± 1.4,23.5 ± 2.6,"8,990","179,800"
BL2 (BF),42.7 ± 2.8,47.0 ± 1.6,44.9 ± 2.1,"8,126","162,520"
BL3 (Mixte),50.3 ± 2.5,48.8 ± 2.3,49.6 ± 2.4,"17,116","342,320"
S1 Transfer,51.0 ± 1.6,44.7 ± 1.3,47.9 ± 1.5,"17,116","153,160"
S1 +Optuna,49.8 ± 1.1,45.4 ± 1.5,47.6 ± 1.3,"17,116","162,222"
S2 CoTrain,37.5 ± 0.8,37.4 ± 0.3,37.5 ± 0.2,"17,116","183,040"
S2 +Optuna,42.2 ± 0.5,41.6 ± 0.3,41.9 ± 0.4,"17,116","259,080"
S3 Curriculum,47.5 ± 1.4,46.1 ± 1.1,46.8 ± 1.2,"17,116","158,880"
S3 +Optuna,52.8 ± 0.4,51.8 ± 0.6,52.3 ± 0.5,"17,116","256,312"


### Intel

,Test HF,Test BF,Test Mixte,Coût données,Coût total
Modèle,,,,,
BL1 (HF),75.8 ± 2.2,47.0 ± 0.8,61.4 ± 1.1,"14,020","280,400"
BL2 (BF),78.7 ± 2.7,81.9 ± 1.4,80.3 ± 1.4,"12,632","252,640"
BL3 (Mixte),82.8 ± 1.2,82.1 ± 1.2,82.5 ± 1.2,"26,652","533,040"
S1 Transfer,86.1 ± 0.4,79.1 ± 1.4,82.6 ± 0.7,"26,652","238,420"
S1 +Optuna,86.0 ± 0.4,80.0 ± 0.4,83.0 ± 0.1,"26,652","201,946"
S2 CoTrain,82.0 ± 1.0,79.8 ± 0.9,80.9 ± 0.9,"26,652","291,200"
S2 +Optuna,82.0 ± 1.2,80.2 ± 2.1,81.1 ± 1.1,"26,652","294,520"
S3 Curriculum,84.6 ± 1.0,82.6 ± 1.5,83.6 ± 1.2,"26,652","250,880"
S3 +Optuna,86.0 ± 0.8,84.4 ± 0.3,85.2 ± 0.5,"26,652","368,074"


CSV sauvés dans C:\VSCode\PF22_MultiFidelity_DL\results_gpu_run\analysis


## 3) Précision Test HF par modèle et dataset

In [3]:
n = len(A.DATASETS)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), sharey=True)
if n == 1:
    axes = [axes]
for ax, ds in zip(axes, A.DATASETS):
    sub = df[df.dataset == ds]
    ax.bar(range(len(sub)), sub.HF, yerr=sub.HF_std, capsize=2,
           color=list(sub.color), edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub.model, rotation=60, ha="right", fontsize=8)
    ax.set_title(ds)
    ax.set_ylabel("Test HF (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Précision Test HF par modèle (moyenne ± std, 3 seeds)", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(OUT, "hf_par_modele.png"), dpi=160, bbox_inches="tight")
plt.show()

C:\Users\ivann\AppData\Local\Temp\ipykernel_23248\4081206134.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4) Frontière de Pareto — coût total vs précision HF

*Haut-gauche = mieux (haute précision, faible coût).*

In [4]:
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
if n == 1:
    axes = [axes]
for ax, ds in zip(axes, A.DATASETS):
    sub = df[df.dataset == ds]
    ax.scatter(sub.total_cost, sub.HF, c=list(sub.color), s=90, edgecolor="black", zorder=3)
    for _, r in sub.iterrows():
        ax.annotate(r.model, (r.total_cost, r.HF), fontsize=6.5,
                    xytext=(4, 3), textcoords="offset points")
    ax.set_title(ds)
    ax.set_xlabel("Coût total (CA)")
    ax.set_ylabel("Test HF (%)")
    ax.grid(alpha=0.3)
fig.suptitle("Pareto : coût total vs précision HF", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(OUT, "pareto_cout_hf.png"), dpi=160, bbox_inches="tight")
plt.show()

C:\Users\ivann\AppData\Local\Temp\ipykernel_23248\4023876740.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5) Classement global (rang moyen sur HF+BF+Mixte × 3 datasets)

In [5]:
parts = []
for ds in A.DATASETS:
    sub = df[df.dataset == ds].copy()
    for met in ["HF", "BF", "Mixte"]:
        sub[f"rank_{met}"] = sub[met].rank(ascending=False, method="min")
    parts.append(sub)
allr = pd.concat(parts)
avg = (allr.groupby("model")[["rank_HF", "rank_BF", "rank_Mixte"]].mean()
       .mean(axis=1).sort_values())
colors = [df[df.model == m].iloc[0].color for m in avg.index]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(list(avg.index)[::-1], list(avg.values)[::-1],
        color=colors[::-1], edgecolor="black")
for i, v in enumerate(list(avg.values)[::-1]):
    ax.text(v + 0.03, i, f"{v:.2f}", va="center", fontsize=8)
ax.set_xlabel("Rang moyen (1 = meilleur)")
ax.set_title("Classement global des modèles")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "classement_global.png"), dpi=160, bbox_inches="tight")
plt.show()
print(avg.round(2).to_string())

model
S3 +Optuna        1.78
S5 EWC            4.11
S3 Curriculum     4.22
BL3 (Mixte)       4.33
S1 +Optuna        5.00
S1 Transfer       5.22
S5 +Optuna        6.22
BL2 (BF)          7.22
S2 +Optuna        7.67
S2 CoTrain        9.22
BL1 (HF)         11.00


C:\Users\ivann\AppData\Local\Temp\ipykernel_23248\15555436.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6) Meilleurs modèles par dataset

In [6]:
for ds in A.DATASETS:
    sub = df[df.dataset == ds]
    bh = sub.loc[sub.HF.idxmax()]
    bm = sub.loc[sub.Mixte.idxmax()]
    print(f"{ds:11s} | meilleur HF: {bh.model:14s} {bh.HF:5.1f}%  | "
          f"meilleur Mixte: {bm.model:14s} {bm.Mixte:5.1f}%")

Animals-10  | meilleur HF: S1 +Optuna      78.1%  | meilleur Mixte: S3 +Optuna      75.4%
Imagewoof   | meilleur HF: S5 +Optuna      53.4%  | meilleur Mixte: S3 +Optuna      52.3%
Intel       | meilleur HF: S1 Transfer     86.1%  | meilleur Mixte: S3 +Optuna      85.2%
